In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

## 数据导入
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")


#-------------------------------------------------------------------------------------------------------------------------------------#
#                                                              数据处理                                                                #
#-------------------------------------------------------------------------------------------------------------------------------------#
## 缺失值处理
train_data['Age'] = train_data['Age'].fillna(train_data['Age'].median())
test_data['Age'] = test_data['Age'].fillna(test_data['Age'].median())
train_data['Fare'] = train_data['Fare'].fillna(train_data['Fare'].median())
test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].median())

## 分箱
bins = [0, 12, 18, 60, 100]
labels = [0, 1, 2, 3]
train_data['Age_Bin'] = pd.cut(train_data['Age'], bins=bins, labels=labels)
test_data['Age_Bin'] = pd.cut(test_data['Age'], bins=bins, labels=labels)

## 家庭大小 
train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1
train_data['IsAlone'] = (train_data['FamilySize'] == 1).astype(int)
test_data['IsAlone'] = (test_data['FamilySize'] == 1).astype(int)
#---------------------------------------------------------------------------------------------------------------------------------------#
#                                                                                                                                       #
#---------------------------------------------------------------------------------------------------------------------------------------#


##特征提取
features = ["Pclass", "Sex", "Age_Bin", "FamilySize","IsAlone"]
##维度轴数+维度对齐
x = pd.get_dummies(train_data[features])
y = train_data["Survived"]
x_test = pd.get_dummies(test_data[features])
x_test = x_test.reindex(columns=x.columns, fill_value=0)


##模型设置
model = RandomForestClassifier(n_estimators=100,random_state=91)
model.fit(x,y)
predictions = model.predict(x_test)

##输出
submission = pd.DataFrame({"PassengerId":test_data["PassengerId"],"Survived":predictions})
submission.to_csv("submission.csv", index=False)